In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm

import kagglehub

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here

# Find dataset root
dataset_root = path
candidates = [
    os.path.join(path, "q1-stage-3-2026"),
    os.path.join(path, "dataset"),
    os.path.join(path, "data"),
    os.path.join(path, "Potato"),
    os.path.join(path, "PlantVillage"),
]
for c in candidates:
    if os.path.isdir(c):
        dataset_root = c
        break

# If there is a single folder inside, prefer it
subdirs = [d for d in os.listdir(dataset_root) if os.path.isdir(os.path.join(dataset_root, d))]
if len(subdirs) == 1:
    dataset_root = os.path.join(dataset_root, subdirs[0])

print("Using dataset root:", dataset_root)
print("Root contents:", os.listdir(dataset_root)[:10])

# Identify train/test if present, otherwise split from one folder
train_dir = None
test_dir = None

for name in ["train", "training", "Train", "Training"]:
    p = os.path.join(dataset_root, name)
    if os.path.isdir(p):
        train_dir = p
        break

for name in ["test", "testing", "Test", "Testing", "val", "valid", "validation", "Val", "Validation"]:
    p = os.path.join(dataset_root, name)
    if os.path.isdir(p):
        test_dir = p
        break

# Transforms
# Use RandomRotation(15) on train + Resize to 32x32
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

# Create datasets
if train_dir is not None and test_dir is not None:
    train_dataset = ImageFolder(root=train_dir, transform=train_transform)
    test_dataset  = ImageFolder(root=test_dir,  transform=test_transform)
else:
    full_dataset = ImageFolder(root=dataset_root, transform=train_transform)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

    # Make test use testtransform (Subset keeps base dataset; update safely)
    if hasattr(train_dataset, "dataset"):
        train_dataset.dataset.transform = train_transform
    if hasattr(test_dataset, "dataset"):
        test_dataset.dataset.transform = test_transform

# Class info
if hasattr(train_dataset, "classes"):
    classes = train_dataset.classes
elif hasattr(train_dataset, "dataset") and hasattr(train_dataset.dataset, "classes"):
    classes = train_dataset.dataset.classes
else:
    classes = ["Early_blight", "Late_blight", "healthy"]

num_classes = 3
print("Classes:", classes)
print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

# DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Display samples
def show_samples(dataset, classes, n=8):
    fig, axes = plt.subplots(2, n//2, figsize=(14, 5))
    axes = axes.flatten()
    for i in range(n):
        x, y = dataset[i]
        axes[i].imshow(x.permute(1, 2, 0))
        axes[i].set_title(classes[int(y)] if int(y) < len(classes) else str(int(y)))
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()

show_samples(train_dataset, classes, n=8)



In [ ]:
# Write your code here

class CNN5(nn.Module):
    def __init__(self, num_classes=3, use_bn=True):
        super().__init__()

        def block(in_ch, out_ch):
            layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)]
            if use_bn:
                layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            return nn.Sequential(*layers)

        self.conv1 = block(3, 32)
        self.conv2 = block(32, 64)
        self.conv3 = block(64, 128)
        self.conv4 = block(128, 128)
        self.conv5 = block(128, 256)

        self.pool = nn.MaxPool2d(2, 2)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.pool(self.conv1(x))   # 32 -> 16
        x = self.pool(self.conv2(x))   # 16 -> 8
        x = self.pool(self.conv3(x))   # 8 -> 4
        x = self.pool(self.conv4(x))   # 4 -> 2
        x = self.pool(self.conv5(x))   # 2 -> 1
        x = self.classifier(x)
        return x

In [ ]:
# Write your code here
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Train", leave=False):
        images = images.to(device)
        labels = labels.to(device).long()

        # TODO: Complete the training step
        # 1. Forward pass
        # 2. Compute loss
        # 3. Zero gradients
        # 4. Backward pass
        # 5. Update weights

        # YOUR CODE HERE
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Val", leave=False):
        images = images.to(device)
        labels = labels.to(device).long()

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

In [ ]:
# Write your code here
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = CNN5(num_classes=3, use_bn=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = validate(model, test_loader, criterion, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_accs.append(tr_acc)
    val_accs.append(va_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss = {tr_loss:.4f}, Val Loss = {va_loss:.4f}, "
          f"Train Acc = {tr_acc:.4f}, Val Acc = {va_acc:.4f}")

# Plot losses
plt.figure(figsize=(10, 4))
plt.plot(range(1, num_epochs+1), train_losses, marker="o", label="Train Loss")
plt.plot(range(1, num_epochs+1), val_losses, marker="o", label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

# Plot accuracy
plt.figure(figsize=(10, 4))
plt.plot(range(1, num_epochs+1), train_accs, marker="o", label="Train Acc")
plt.plot(range(1, num_epochs+1), val_accs, marker="o", label="Val Acc")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Write your code here
class CNN5Residual(nn.Module):
    def __init__(self, num_classes=3, use_bn=True):
        super().__init__()

        def block(in_ch, out_ch):
            layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)]
            if use_bn:
                layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            return nn.Sequential(*layers)

        self.conv1 = block(3, 32)
        self.conv2 = block(32, 64)      # skip from here
        self.conv3 = block(64, 128)
        self.conv4 = block(128, 128)    # to here
        self.conv5 = block(128, 256)

        self.pool = nn.MaxPool2d(2, 2)

        # match skip channels to con4 output channels 128 for summation
        self.skip_proj = nn.Conv2d(64, 128, kernel_size=1)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.pool(self.conv1(x))     # 32 -> 16
        x = self.pool(self.conv2(x))     # 16 -> 8
        skip = x                         # skip at 8x8 64ch

        x = self.pool(self.conv3(x))     # 8 -> 4
        x = self.pool(self.conv4(x))     # 4 -> 2

        # Resize skip to 2x2 then match channels then sum
        skip = nn.functional.adaptive_avg_pool2d(skip, (2, 2))  # 8x8 -> 2x2
        skip = self.skip_proj(skip)                             # 64ch -> 128ch
        x = x + skip

        x = self.pool(self.conv5(x))     # 2 -> 1
        x = self.classifier(x)
        return x

# Retrain residual model
res_model = CNN5Residual(num_classes=3, use_bn=True).to(device)
res_criterion = nn.CrossEntropyLoss()
res_optimizer = optim.Adam(res_model.parameters(), lr=1e-3)

res_train_losses = []
res_val_losses = []
res_train_accs = []
res_val_accs = []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_one_epoch(res_model, train_loader, res_criterion, res_optimizer, device)
    va_loss, va_acc = validate(res_model, test_loader, res_criterion, device)

    res_train_losses.append(tr_loss)
    res_val_losses.append(va_loss)
    res_train_accs.append(tr_acc)
    res_val_accs.append(va_acc)

    print(f"[RES] Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss = {tr_loss:.4f}, Val Loss = {va_loss:.4f}, "
          f"Train Acc = {tr_acc:.4f}, Val Acc = {va_acc:.4f}")

# Ploting residual losses
plt.figure(figsize=(10, 4))
plt.plot(range(1, num_epochs+1), res_train_losses, marker="o", label="Train Loss")
plt.plot(range(1, num_epochs+1), res_val_losses, marker="o", label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Residual model: training and validation accuracy")
plt.legend()
plt.grid(True)
plt.show()

# Ploiting residual accuracy
plt.figure(figsize=(10, 4))
plt.plot(range(1, num_epochs+1), res_train_accs, marker="o", label="Train Acc")
plt.plot(range(1, num_epochs+1), res_val_accs, marker="o", label="Val Acc")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Residual model: training and validation accuracy")
plt.legend()
plt.grid(True)
plt.show()